In [4]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
from PIL import Image  # For GIF creation
import os
import warnings
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from itertools import combinations
from tqdm import tqdm

import numpy as np
import pandas as pd
import skimage
import torch

import neuralpredictors

warnings.filterwarnings("ignore")

from nnfabrik.builder import get_data, get_model, get_trainer
from nnfabrik.utility.nn_helpers import set_random_seed

In [ ]:
def ari_baseline(keys1):    
    """ For FM score or v score just replace ari 
    ari_score = fowlkes_mallows_score(
                        predictions_dict[k1][seed1], 
                        predictions_dict[k1][seed2]
                    )
    v_score = homogeneity_completeness_v_measure
    """
    seeds=[10,42,100]
    ari_results = {}         # Stores ARI scores against ground truth
    predictions_dict = {}    

    for k1 in keys1:
        predictions_dict[k1] = {}
        cluster=k1
        for seed in seeds:
            predictions_dict[k1][seed] = np.load(f'ENTER PATHnpy')

    ari_results = {}  

    for k1 in keys1:
        ari_results[k1] = {}  
        for seed1, seed2 in combinations(seeds, 2):
            try:
                # Check if both seeds exist before computing ARI
                if (seed1 in predictions_dict[k1]and 
                    seed2 in predictions_dict[k1]):
                    ari_score = adjusted_rand_score(
                        predictions_dict[k1][seed1], 
                        predictions_dict[k1][seed2]
                    )
                else:
                    ari_score = np.nan  
            except Exception as e:
                ari_score = np.nan  
            ari_results[k1][(seed1, seed2)] = ari_score


    ari_values = [list(ari_results[k1].values()) for k1 in keys1]
    ari_means = np.array([np.mean([v for v in vals if v is not None]) if any(v is not None for v in vals) else None for vals in ari_values])
    ari_stds = np.array([np.std([v for v in vals if v is not None]) if any(v is not None for v in vals) else None for vals in ari_values])

    return ari_means, ari_stds